# Module 15 — Logistic Regression Practice Problems
### Titanic Survival Prediction

এই notebook-এ Module 15-এর প্রতিটি topic-এর উপর হাতে-কলমে practice করবে।
প্রতিটি problem-এর নিচে code cell-এ solution লিখো।

**Dataset:** `titanic_data_updated.csv`

---
### Problem 1 — Imports and Data Loading

নিচের সব library import করো:
- `numpy`, `pandas`
- `sklearn` থেকে: `train_test_split`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder`, `LabelEncoder`, `StandardScaler`, `MinMaxScaler`, `Pipeline`, `ColumnTransformer`, `LogisticRegression`

`titanic_data_updated.csv` লোড করো `df` নামে এবং প্রথম 5টি row দেখাও।

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [6]:
df = pd.read_csv("F:/Jupyter Projects/Phitron AL ML/Dataset/Titanic-Dataset.csv")
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


---
### Problem 2 — Feature Engineering

দুটি নতুন column তৈরি করো:

1. `Family_Size` — formula: `SibSp + Parch + 1`
2. `Deck` — `Cabin` column-এর NaN গুলো `"Missing"` দিয়ে fill করো, তারপর প্রথম character নিয়ে `Deck` column তৈরি করো

শেষে `df.sample(5)` দিয়ে result দেখাও।

In [7]:
# YOUR CODE HERE

df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Deck'] = df['Cabin'].fillna('Missing').astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
720,721,1,2,"Harper, Miss. Annie Jessie ""Nina""",female,6.0,0,1,248727,33.0000,NaN,S,2,M
336,337,0,1,"Pears, Mr. Thomas Clinton",male,29.0,1,0,113776,66.6000,C2,S,2,C
110,111,0,1,"Porter, Mr. Walter Chamberlain",male,47.0,0,0,110465,52.0000,C110,S,1,C
442,443,0,3,"Petterson, Mr. Johan Emil",male,25.0,1,0,347076,7.7750,NaN,S,2,M
737,738,1,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,B101,C,1,B


---
### Problem 3 — X and y Split

`df` থেকে features এবং target আলাদা করো:

- `X` = `Survived` বাদে সব column
- `y` = শুধু `Survived` column

`X.shape` এবং `y.shape` print করো।

In [8]:
# YOUR CODE HERE
X = df.drop(columns = ['Survived'])
y = df['Survived']

X.shape, y.shape

((891, 13), (891,))

In [9]:
X.sample(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
196,197,3,"Mernagh, Mr. Robert",male,NaN,0,0,368703,7.75,NaN,Q,1,M
11,12,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.55,C103,S,1,C
706,707,2,"Kelly, Mrs. Florence ""Fannie""",female,45.0,0,0,223596,13.50,NaN,S,1,M
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,NaN,S,1,M
27,28,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.00,C23 C25 C27,S,6,C


In [10]:
y.sample(5)

571    1
685    0
497    0
673    1
548    0
Name: Survived, dtype: int64

---
### Problem 4 — Train-Test Split

`train_test_split` ব্যবহার করে `X_train`, `X_test`, `y_train`, `y_test` তৈরি করো।

Parameters:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

`X_train` এবং `X_test`-এর shape print করো।

In [11]:
# YOUR CODE HERE
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)
X_train.shape, X_test.shape

((712, 13), (179, 13))

---
### Problem 5 — Outlier Handling

**Age column — Z-score method (rows remove করো):**
- `Z_score = (Age - mean) / std`
- যেসব row-এ `|Z_score| > 3`, সেগুলো `X_train` এবং `y_train` থেকে বাদ দাও

**Fare column — IQR clipping (rows রাখো, values clip করো):**
- `Q1` = 25th percentile, `Q3` = 75th percentile
- `IQR = Q3 - Q1`
- `minimum = max(0, Q1 - 1.5 * IQR)`
- `maximum = Q3 + 1.5 * IQR`
- `clip()` দিয়ে Fare column-কে `[minimum, maximum]` range-এ রাখো

In [12]:
# YOUR CODE HERE

Z_score = (X_train['Age'] - X_train['Age'].mean()) / X_train['Age'].std()
X_train = X_train[abs(Z_score) <= 3]
y_train = y_train[abs(Z_score) <= 3]

q1 = X_train['Fare'].quantile(0.25)
q3 = X_train['Fare'].quantile(0.75)
iqr = q3 - q1
minimum = max(0, q1 - 1.5 * iqr)
maximum = q3 + 1.5 * iqr
X_train['Fare'] = X_train['Fare'].clip(lower = minimum, upper = maximum)

---
### Problem 6 — Numerical Pipelines

দুটি numerical pipeline তৈরি করো:

**p1** — `Age` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='mean')`
- Step 2: `StandardScaler()`

**p2** — `Fare` এবং `Family_Size` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='median')`
- Step 2: `MinMaxScaler()`

In [13]:
# YOUR CODE HERE

age_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'mean')),
    ('scaler', StandardScaler())
])

fare_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', MinMaxScaler())
])

---
### Problem 7 — Categorical Pipelines and ColumnTransformer

দুটি categorical pipeline তৈরি করো:

**p3** — `Embarked`, `Sex`, `Deck` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')`

**p4** — `Pclass` column-এর জন্য:
- `categories = [['third', 'second', 'first']]`
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OrdinalEncoder(categories=categories)`
- Step 3: `MinMaxScaler()`

তারপর `ColumnTransformer` দিয়ে `preprocessor` তৈরি করো:
- `pipeline_1` → `p1` → `['Age']`
- `pipeline_2` → `p2` → `['Fare', 'Family_Size']`
- `pipeline_3` → `p3` → `['Embarked', 'Sex', 'Deck']`
- `pipeline_4` → `p4` → `['Pclass']`
- `remainder='drop'`

In [14]:
# YOUR CODE HERE
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OneHotEncoder(sparse_output = False, drop = 'first', handle_unknown = 'ignore'))
])

# categories = [['third','second','first']]
categorical_transformer2 = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    # ('encoder', OrdinalEncoder(categories = categories)),
    ('scaler', MinMaxScaler())
])


preprocessor = ColumnTransformer(transformers = [
    ('pipeline_1', age_transformer, ['Age']),
    ('pipeline_2', fare_transformer, ['Fare', 'Family_Size']),
    ('pipeline_3', categorical_transformer, ['Embarked', 'Sex', 'Deck']),
    ('pipeline_4', categorical_transformer2, ['Pclass'])
], remainder = 'drop')

preprocessor

,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


---
### Problem 8 — Target Column Encoding

`y_train` এবং `y_test`-এ এখন `"yes"` এবং `"no"` string আছে। Model train করার আগে এগুলো numeric করতে হবে।

- `LabelEncoder` দিয়ে `y_train` এবং `y_test` encode করো
- encode করার আগে এবং পরে unique values print করো

In [15]:
# YOUR CODE HERE
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print(np.unique(y_train))
print(np.unique(y_test))

[0 1]
[0 1]


---
### Problem 9 — Model Training and Prediction

`lr_model` নামে একটি final `Pipeline` তৈরি করো:
- Step 1: `preprocessor` (Problem 7-এ তৈরি করা)
- Step 2: `LogisticRegression(class_weight='balanced', max_iter=1000)`

তারপর:
- `lr_model.fit()` দিয়ে `X_train` এবং `y_train` দিয়ে model train করো
- `predict()` দিয়ে `X_test`-এর prediction করো, `y_pred` নামে save করো
- প্রথম 10টি prediction print করো

In [16]:
# YOUR CODE HERE
lr_model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced',max_iter=1000))
])

lr_model

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
print(y_pred[:10])

[0 0 0 0 1 1 1 0 1 0]


---
### Problem 10 — Evaluation

`sklearn.metrics` থেকে `accuracy_score`, `precision_score`, `recall_score` import করো।

`y_test` এবং `y_pred` দিয়ে তিনটি score calculate করো এবং নিচের format-এ print করো:

```
Accuracy  : 0.xx
Precision : 0.xx
Recall    : 0.xx
```

In [18]:
# YOUR CODE HERE

from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f'Accuracy : {accuracy:.2f}')
print(f'Precision : {precision:.2f}')
print(f'Recall : {recall:.2f}')

Accuracy : 0.77
Precision : 0.68
Recall : 0.75


In [19]:
# YOUR CODE HERE
dc_model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('decision', DecisionTreeClassifier(class_weight='balanced'))
])

dc_model

,steps,"[('preprocessor', ...), ('decision', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
dc_model.fit(X_train, y_train)
y_pred = dc_model.predict(X_test)

accuracy_score(y_test, y_pred)

0.7318435754189944